In [1]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# ============================================================
# 0. 데이터 불러오기: 2-5 train_test_comparison에서 제거된 이상치까지 반영된 데이터셋
# ============================================================

train_full = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '0_train_full_clean.parquet')
test = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '0_test_future.parquet')

print(train_full.shape, test.shape)

from sklearn import set_config
set_config(display='text')
# scikit-learn의 HTML 출력 과정에서 발생한 cp949 간 UTF-8 충돌 때문에 넣는 코드

(749609, 34) (114111, 34)


In [2]:
# ============================================================
# 1-3. 최종 컬럼군 정의
# ============================================================

# 종속변수군
target_col = 'target_maxrise_6h'

# 독립변수군
feature_cols = [
    col for col in train_full.columns
    if col not in ('timestamp', target_col)
]

assert len(feature_cols) == 32, (
    f'독립변수 개수 이상: 예상 32개, 현재 {len(feature_cols)}개'
)

## 3. 강우사상 라벨링, GroupKFold 5-Fold 배정(CSV로 저장)

In [3]:
# ============================================================
# 3-1. 강우사상별 데이터 분할 -> event_id 컬럼 생성
# ============================================================

# 전체 timestamp 추출(중복없이), 시간순 정렬
event_time = (
    train_full[['timestamp']] 
    .drop_duplicates() 
    .sort_values('timestamp') 
    .reset_index(drop = True) 
)

# 차분: diff 계산
event_time['event_time_diff'] = (
    event_time['timestamp']
    .diff()
)

# 차분값이 1시간이 아닌 경우를 event 경계로 추출
cond_split_time = (
    event_time['event_time_diff'] != pd.Timedelta(hours=1)
)

# 강우사상 시간대별 event_id 부여
event_time['event_id'] = cond_split_time.cumsum()

# 시험처가 제시한 강우사상 개수와 맞는지 확인: 122개
assert event_time['event_id'].nunique() == 122, (
    f"주최측이 제시한 강우사상 122개, 현재 {event_time['event_id'].nunique()}개, 오류"
)

# train_full에 event_id 컬럼 붙이기(매핑)
rule_event_id_map = event_time.set_index('timestamp')['event_id']  # 매핑규칙(조인대상 인덱스 필수)

train_full['event_id'] = train_full['timestamp'].map(rule_event_id_map)

assert not train_full['event_id'].isnull().any(), (
    'event_id 매핑 누락 발생'
)

## 4. Tidal, Normal 데이터 분할

In [4]:
# ============================================================
# 4-1. Tidal 데이터 필터링
# ============================================================

# 조건 설정
tidal_cond = (
    (train_full['wl_diff_6h'] < -1) &  # 수위하강이 발생하면서(0이 아닌)
    ((train_full[target_col] + train_full['wl_diff_6h']).abs() < 2)  # 수위회복 패턴이 뒤이어 발생하는 조건
)

# 원본 station_id 결측치 검증
assert not train_full['station_id'].isnull().any(), "station_id 결측치 존재"

tidal_station_df = pd.DataFrame({
    'total_count' : train_full['station_id'].value_counts(),
    'tidal_count' : train_full.loc[tidal_cond, 'station_id'].value_counts()
}).fillna(0).astype(int)

# 한번 더 검증
assert not tidal_station_df.isnull().any().any(), "아직 NaN 있음"

# 관측소별 tidal 비율 추가
tidal_station_df['tidal_ratio(%)'] = (
    tidal_station_df['tidal_count'] / tidal_station_df['total_count'] * 100
)

# tidal_count기준 내림차순 정렬
tidal_station_df = tidal_station_df.sort_values(
    'tidal_count', ascending=False
)

tidal_stations = (
    tidal_station_df
    .head(9)  # 기존 Train(Validation 제외)에서 Tidal 발생 비율 10% 초과였던 9개 관측소 유지
    .index
)

print(tidal_stations)

Index(['ST0326', 'ST0358', 'ST0292', 'ST0347', 'ST0159', 'ST0110', 'ST0415',
       'ST0313', 'ST0234'],
      dtype='object', name='station_id')


In [5]:
# ============================================================
# 4-2. Tidal / Normal 데이터 분리
# ============================================================

# 4-1. Train 분리
tidal_train = train_full.loc[
    train_full['station_id'].isin(tidal_stations)
].copy()

normal_train = train_full.loc[
    ~train_full['station_id'].isin(tidal_stations)
].copy()

# 4-2. Test 분리
tidal_test = test.loc[
    test['station_id'].isin(tidal_stations)
].copy()

normal_test = test.loc[
    ~test['station_id'].isin(tidal_stations)
].copy()

# 4-3. 분리 결과 확인
print('<Train 분리 결과>')
print(f'Train 전체 : {len(train_full):,}')
print(f'Tidal      : {len(tidal_train):,}')
print(f'Normal     : {len(normal_train):,}')

print('\n<Test 분리 결과>')
print(f'Test 전체  : {len(test):,}')
print(f'Tidal      : {len(tidal_test):,}')
print(f'Normal     : {len(normal_test):,}')

# 4-4. 데이터 분리 검증
assert len(tidal_train) + len(normal_train) == len(train_full)
assert len(tidal_test) + len(normal_test) == len(test)

print('\n데이터 분리 검증 완료')

<Train 분리 결과>
Train 전체 : 749,609
Tidal      : 14,251
Normal     : 735,358

<Test 분리 결과>
Test 전체  : 114,111
Tidal      : 2,169
Normal     : 111,942

데이터 분리 검증 완료


## 5. 독립변수 / 종속변수 설정

In [ ]:
# ============================================================
# 5. 독립변수 / 종속변수 분리
# ============================================================

assert len(feature_cols) == 32, (
    f'독립변수 개수 이상: 예상 32개, 현재 {len(feature_cols)}개'
)

# Tidal
X_tidal_train = tidal_train[feature_cols].copy()
X_tidal_test = tidal_test[feature_cols].copy()

y_tidal_train = tidal_train[target_col]

# Normal
X_normal_train = normal_train[feature_cols].copy()
X_normal_test = normal_test[feature_cols].copy()

y_normal_train = normal_train[target_col]

# 데이터 분리 검증
assert X_tidal_train.index.equals(y_tidal_train.index)
assert X_normal_train.index.equals(y_normal_train.index)

assert X_tidal_train.columns.equals(X_tidal_test.columns)
assert X_normal_train.columns.equals(X_normal_test.columns)

print('독립변수 / 종속변수 분리 검증 완료')

독립변수 / 종속변수 분리 검증 완료


## 6. Tidal 데이터셋 모델 생성: 단일 홀드아웃, CatBoost, ExtraTree 앙상블

In [7]:
# ============================================================
# 6-1. CatBoost에 넣을 범주형 변수 처리
# ============================================================

for X in [
    X_tidal_train, X_tidal_test,
    X_normal_train, X_normal_test
]:
    X['station_id'] = X['station_id'].astype(str)

In [8]:
# ============================================================
# 6-2. CatBoost 모델로 tidal 예측
# ============================================================
from catboost import CatBoostRegressor

# Tidal CatBoost 모델

cat_tidal = CatBoostRegressor(
    iterations=2976,  # train tidal 5-Fold best rmse의 best iteration
    depth=9,
    learning_rate=0.04809875432044185,
    l2_leaf_reg=0.27010527749605473,
    random_strength=0.005404103854647328,
    random_state=42,
    cat_features=['station_id'],
    verbose=200
)

cat_tidal.fit(
    X_tidal_train, y_tidal_train,
)

pred_cat_tidal = cat_tidal.predict(X_tidal_test)

# # Tidal CatBoost 모델 저장
# cat_tidal.save_model(
#     str(PROJECT_ROOT / 'model' / 'cat_tidal.cbm')
# )


0:	learn: 1.6888800	total: 212ms	remaining: 10m 31s
200:	learn: 0.3008306	total: 11.4s	remaining: 2m 37s
400:	learn: 0.2313446	total: 22.3s	remaining: 2m 23s
600:	learn: 0.1866076	total: 33.7s	remaining: 2m 13s
800:	learn: 0.1565880	total: 44.7s	remaining: 2m 1s
1000:	learn: 0.1325576	total: 56.6s	remaining: 1m 51s
1200:	learn: 0.1138963	total: 1m 7s	remaining: 1m 39s
1400:	learn: 0.0986999	total: 1m 19s	remaining: 1m 29s
1600:	learn: 0.0870864	total: 1m 32s	remaining: 1m 19s
1800:	learn: 0.0760648	total: 1m 45s	remaining: 1m 9s
2000:	learn: 0.0678037	total: 1m 57s	remaining: 57.4s
2200:	learn: 0.0599734	total: 2m 9s	remaining: 45.5s
2400:	learn: 0.0537723	total: 2m 19s	remaining: 33.5s
2600:	learn: 0.0481607	total: 2m 30s	remaining: 21.7s
2800:	learn: 0.0431536	total: 2m 43s	remaining: 10.2s
2975:	learn: 0.0393469	total: 2m 55s	remaining: 0us


In [10]:
# ============================================================
# 6-3. ExtraTrees 모델에 넣을 범주형 변수 처리: OneHotEncoding
# ============================================================

from sklearn.preprocessing import OneHotEncoder

# OneHotEncoder 정의 및 학습
ohe = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)

ohe.fit(X_tidal_train[['station_id']])

# Train / Test 원핫인코딩
X_tidal_train_ohe = pd.concat([
    X_tidal_train.drop(columns='station_id'),  # 기존에 있던 'station_id' 컬럼은 제거
    pd.DataFrame(  # 원핫인코딩 결과 데이터프레임으로 만들어서 붙이기
        ohe.transform(X_tidal_train[['station_id']]),  
        columns=ohe.get_feature_names_out(),  # 컬럼 이름을 해당 station_id로 받아오기
        index=X_tidal_train.index
    )
], axis=1)

X_tidal_test_ohe = pd.concat([
    X_tidal_test.drop(columns='station_id'),
    pd.DataFrame(
        ohe.transform(X_tidal_test[['station_id']]),
        columns=ohe.get_feature_names_out(),
        index=X_tidal_test.index
    )
], axis=1)

# 검증
assert 'station_id' not in X_tidal_train_ohe.columns
assert X_tidal_train_ohe.columns.equals(X_tidal_test_ohe.columns)
assert X_tidal_train_ohe.index.equals(y_tidal_train.index)

# 원핫인코딩 전후 Shape 비교
print(f'Tidal Train : {X_tidal_train.shape} -> {X_tidal_train_ohe.shape}')
print(f'Tidal Test  : {X_tidal_test.shape} -> {X_tidal_test_ohe.shape}')

# 학습된 OneHotEncoder 저장
import joblib

# joblib.dump(
#     ohe,
#     PROJECT_ROOT / 'model' / 'ohe_tidal_station_id.joblib'
# )

Tidal Train : (14251, 32) -> (14251, 40)
Tidal Test  : (2169, 32) -> (2169, 40)


In [11]:
# ============================================================
# 6-4. ExtraTrees 모델로 tidal 예측
# ============================================================
from sklearn.ensemble import ExtraTreesRegressor

# Tidal ExtraTrees 모델
tidal_et_params = {
    'n_estimators': 200,
    'max_depth': None,
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'max_features': 0.7843011071485029
}

et_tidal = ExtraTreesRegressor(
    **tidal_et_params,
    random_state=42,
    n_jobs=-1
)


et_tidal.fit(
    X_tidal_train_ohe, y_tidal_train,
)

pred_et_tidal = et_tidal.predict(X_tidal_test_ohe)

# Tidal ExtraTrees 모델 저장
import joblib

# joblib.dump(
#     et_tidal,
#     PROJECT_ROOT / 'model' / 'et_tidal_ExtraTrees.joblib'
# )

In [12]:
# ============================================================
# 6-5. CatBoost 45% + ExtraTrees 55% 앙상블 예측
# ============================================================

pred_ensemble_tidal = pd.Series(
    pred_cat_tidal * 0.45 + pred_et_tidal * 0.55,
    index=X_tidal_test.index
)

# 앙상블 예측 결과 검증
assert pred_ensemble_tidal.index.equals(X_tidal_test.index)
assert len(pred_ensemble_tidal) == len(tidal_test)
assert pred_ensemble_tidal.notna().all()

print(f'Tidal Test: {len(tidal_test):,}행')
print(f'앙상블 예측: {len(pred_ensemble_tidal):,}행')
print('Tidal 앙상블 예측 검증 완료')

Tidal Test: 2,169행
앙상블 예측: 2,169행
Tidal 앙상블 예측 검증 완료


In [14]:
print('CatBoost RMSE:',
      root_mean_squared_error(y_tidal_test, pred_cat_tidal))

print('ExtraTrees RMSE:',
      root_mean_squared_error(y_tidal_test, pred_et_tidal))

print('Ensemble RMSE:',
      root_mean_squared_error(y_tidal_test, pred_ensemble_tidal))

CatBoost RMSE: 0.3590428229181828
ExtraTrees RMSE: 0.347534473189819
Ensemble RMSE: 0.3445574808560735


In [13]:
# ============================================================
# 6-6. Tidal 앙상블 RMSE 평가
# ============================================================

from sklearn.metrics import root_mean_squared_error

y_tidal_test = tidal_test[target_col]

# 실제값과 예측값 인덱스 검증
assert pred_ensemble_tidal.index.equals(y_tidal_test.index)

# RMSE 계산
rmse_ensemble_tidal = root_mean_squared_error(
    y_tidal_test,
    pred_ensemble_tidal
)

print(f'Tidal 앙상블 RMSE: {rmse_ensemble_tidal:.6f}')

Tidal 앙상블 RMSE: 0.344557


## 7. Normal 데이터셋 모델 생성: K_Fold 교차검증, CatBoost

In [16]:
# ============================================================
# 7-1. 고정 Fold CSV 불러오기 및 Train 데이터 매칭
# ============================================================
N_SPLITS = 5

# 고정 Fold 매핑 파일 불러오기
fold_match = pd.read_csv(
    PROJECT_ROOT.parent / 'data' / '0_kf_train_full_match.csv'
)

# event_id 기준으로 Fold 번호 매칭
normal_train['fold_num'] = normal_train['event_id'].map(
    fold_match.set_index('event_id')['fold_num']  # 매핑규칙: 'event_id'로 조인, 'fold_num' 부여
)

# 매칭 검증
assert normal_train['fold_num'].notna().all()
assert normal_train['fold_num'].between(1, N_SPLITS).all()

print('Normal Fold 매칭 완료')

Normal Fold 매칭 완료


In [17]:
# ============================================================
# 7-2. Normal CatBoost 5-Fold 학습 및 예측
# ============================================================

from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error
import numpy as np

# 결과 저장용 변수
normal_oof_pred = pd.Series(
    index=X_normal_train.index,
    dtype=float
)

normal_test_preds = pd.DataFrame(
    index=X_normal_test.index
)

normal_fold_results = []

normal_cat_params = {
    'depth': 9,
    'learning_rate': 0.029074177663135446,
    'l2_leaf_reg': 4.928912168955859,
    'random_strength': 6.6331205928943495
}

# Fold별 학습
for fold in range(1, N_SPLITS + 1):

    print(f'\n========== Normal Fold {fold} ==========')

    # 기존 Fold 번호로 분할
    train_mask = normal_train['fold_num'] != fold
    valid_mask = normal_train['fold_num'] == fold

    X_kf_tr = X_normal_train.loc[train_mask]
    X_kf_val = X_normal_train.loc[valid_mask]

    y_kf_tr = y_normal_train.loc[train_mask]
    y_kf_val = y_normal_train.loc[valid_mask]

    assert len(X_kf_tr) > 0 and len(X_kf_val) > 0, f'Fold {fold}: 학습/검증 데이터 없음'

    # CatBoost 최종 선정모델
    model = CatBoostRegressor(
        **normal_cat_params,
        iterations=1100,  # train 5-Fold에서 fold별 best iteration 편차가 큼 -> 평균 선택
        random_seed=42,
        cat_features=['station_id'],
        thread_count=-1
    )

    model.fit(
        X_kf_tr, y_kf_tr,
        verbose=500
    )

    # Fold별 kf_val 예측
    val_pred = model.predict(X_kf_val)

    # Fold별 OOF 예측 및 RMSE
    normal_oof_pred.loc[X_kf_val.index] = val_pred
    fold_oof_rmse = root_mean_squared_error(y_kf_val, val_pred)

    # Fold별 Test 전체 예측
    test_pred = model.predict(X_normal_test)
    normal_test_preds[f'fold_{fold}'] = test_pred  # Fold별 예측결과가 하나의 컬럼으로 삽입
  
    # Fold 결과 저장
    normal_fold_results.append({
        'fold': fold,
        'train_count': len(X_kf_tr),
        'valid_count': len(X_kf_val),
        'oof_rmse': fold_oof_rmse,
    })

    print(
        f'Fold {fold} | '
        f'OOF RMSE={fold_oof_rmse:.6f} | '
    )

    # # Normal CatBoost 모델 저장
    # model.save_model(
    #     str(PROJECT_ROOT / 'model' / f"cat_{fold}fold_normal.cbm")
    # )


========== Normal Fold 1 ==========
0:	learn: 0.2761066	total: 164ms	remaining: 3m
500:	learn: 0.2006246	total: 1m 22s	remaining: 1m 38s
1000:	learn: 0.1846510	total: 2m 47s	remaining: 16.5s
1099:	learn: 0.1824618	total: 3m 2s	remaining: 0us
Fold 1 | OOF RMSE=0.230210 | 

========== Normal Fold 2 ==========
0:	learn: 0.2937321	total: 157ms	remaining: 2m 52s
500:	learn: 0.2147410	total: 1m 13s	remaining: 1m 27s
1000:	learn: 0.1968354	total: 2m 22s	remaining: 14.1s
1099:	learn: 0.1942761	total: 2m 36s	remaining: 0us
Fold 2 | OOF RMSE=0.164420 | 

========== Normal Fold 3 ==========
0:	learn: 0.2597391	total: 201ms	remaining: 3m 41s
500:	learn: 0.1920092	total: 1m 10s	remaining: 1m 24s
1000:	learn: 0.1764890	total: 2m 19s	remaining: 13.8s
1099:	learn: 0.1744123	total: 2m 33s	remaining: 0us
Fold 3 | OOF RMSE=0.262666 | 

========== Normal Fold 4 ==========
0:	learn: 0.2826555	total: 176ms	remaining: 3m 13s
500:	learn: 0.2047913	total: 1m 11s	remaining: 1m 25s
1000:	learn: 0.1878144	total:

In [ ]:
# ============================================================
# 7-3. Normal CatBoost 학습 및 예측 결과 검증
# ============================================================

# OOF 예측 검증
assert normal_oof_pred.index.equals(y_normal_train.index)
assert normal_oof_pred.notna().all()

# Test 예측 검증
assert normal_test_preds.index.equals(X_normal_test.index)
assert normal_test_preds.shape == (len(X_normal_test), N_SPLITS)
assert normal_test_preds.notna().all().all()

# 전체 OOF RMSE
normal_oof_rmse = root_mean_squared_error(
    y_normal_train, normal_oof_pred
)

print(f'Normal 전체 OOF RMSE: {normal_oof_rmse:.6f}')
print(f'Normal Test 예측: {normal_test_preds.shape}')
print('Normal CatBoost 검증 완료')

Normal 전체 OOF RMSE: 0.217360
Normal Test 예측: (111942, 5)
Normal CatBoost 검증 완료


In [19]:
# ============================================================
# 7-4. Normal CatBoost Test 앙상블 RMSE 평가
# ============================================================

y_normal_test = normal_test[target_col]

# 1. Fold별 Test 예측값 평균
pred_normal_test = normal_test_preds.mean(axis=1)

# 2. 실제값과 예측값 인덱스 검증
assert pred_normal_test.index.equals(y_normal_test.index)

# 3. Test RMSE 계산
rmse_normal_test = root_mean_squared_error(
    y_normal_test,
    pred_normal_test
)

print(f'Normal Test 앙상블 RMSE: {rmse_normal_test:.6f}')

Normal Test 앙상블 RMSE: 0.199533
